In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import os
# Disable all GPUs
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

In [3]:
import os
from datasets import load_dataset, DatasetDict, concatenate_datasets, get_dataset_config_names, load_from_disk
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from typing import Optional, Union
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers
from sentence_transformers.evaluation import InformationRetrievalEvaluator, TripletEvaluator, SequentialEvaluator

/rhome/sawale/miniconda3/envs/slurm-test/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/rhome/sawale/miniconda3/envs/slurm-test/lib/python3.11/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/nas/rhome/sawale/miniconda3/envs/slurm-test/lib/python3.11/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(


# Data on Hugging Face Hub

In [3]:
NROWS = None
val_frac = 0.05
test_frac = 0.05

CACHE_DIR = f"../data/stage1_cache/NROWS_{NROWS}"
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
def split_dataset(
    ds,
    train_split_name: str = "train",
    val_split_name: str = "validation",
    test_split_name: str = "test",
    val_frac: float = 0.05,
    test_frac: float = 0.05,
    seed: int = 42
) -> DatasetDict:
    """
    Always merge all existing splits (if any) into one dataset,
    then carve out a new train/validation split.
    """
    # 1. Merge everything into a single Dataset
    if isinstance(ds, DatasetDict):
        # concatenate all splits (train, validation, test, etc.)
        full_ds = concatenate_datasets(list(ds.values()))
    else:
        # already a single Dataset
        full_ds = ds
        
    # 1.1 Optionally limit the number of rows
    if NROWS:
        # only load a small subset of the dataset
        full_ds = full_ds.select(range(min(NROWS, len(full_ds))))

    # 2) train vs. combined eval
    combined_frac = val_frac + test_frac
    split1 = full_ds.train_test_split(test_size=combined_frac, seed=seed)
    train_ds = split1["train"]
    eval_ds  = split1["test"]

    # 3) validation vs. test
    #    compute relative fraction for validation within eval_ds
    val_relative = val_frac / combined_frac
    split2 = eval_ds.train_test_split(test_size=(1.0 - val_relative), seed=seed)
    val_ds  = split2["train"]
    test_ds = split2["test"]

    return DatasetDict({
        train_split_name: train_ds,
        val_split_name:   val_ds,
        test_split_name:  test_ds,
    })


def get_all_data_subset(name, path, s1, s2, loss_fn):
    configs = get_dataset_config_names(path)
    respo = []

    for cfg in configs:
        _t = {
                "args": {"path": path, "name": cfg},
                "map_fn": lambda ex, s1=s1, s2=s2: {"anchor": ex[s1], "positive": ex[s2]},
                "loss": loss_fn
            }
        respo.append(_t)

    return {f"{name}_{c}": r for r, c in zip(respo, configs)}


dataset_names = {
    "squad_v2": {
        "args": {"path": "rajpurkar/squad_v2"},
        "map_fn": lambda ex: {"anchor": ex["question"], "positive": ex["context"]},
        "loss": MultipleNegativesRankingLoss
    },
    "wikipedia": {
        "args": {"path": "wikimedia/wikipedia", "data_dir": "20231101.en"},
        "map_fn": lambda ex: {"anchor": ex["title"], "positive": ex["text"]},
        "loss": MultipleNegativesRankingLoss
    },
    "StackExchange_Math_titlebody_answer": {
        "args": {"path": "flax-sentence-embeddings/stackexchange_math_jsonl", "data_dir": "titlebody_answer"},
        "map_fn": lambda ex: {"anchor": ex["title"], "positive": ex["upvoted_answer"]},
        "loss": MultipleNegativesRankingLoss
    },
    "StackExchange_Math_title_answer": {
        "args": {"path": "flax-sentence-embeddings/stackexchange_math_jsonl", "data_dir": "title_answer"},
        "map_fn": lambda ex: {"anchor": ex["title"], "positive": ex["upvoted_answer"]},
        "loss": MultipleNegativesRankingLoss
    },
    "StackExchange_title_body": {
        "args": {"path": "flax-sentence-embeddings/stackexchange_title_body_jsonl"},
        "map_fn": lambda ex: {"anchor": ex["texts"][0], "positive": ex["texts"][1]},
        "loss": MultipleNegativesRankingLoss
    },
    "StackExchange_Duplicates_titlebody_titlebody": {
        "args": {"path": "sentence-transformers/stackexchange-duplicates", "data_dir": "post-post-pair"},
        "map_fn": lambda ex: {"anchor": ex["post1"], "positive": ex["post2"]},
        "loss": MultipleNegativesRankingLoss
    },
    "StackExchange_Duplicates_body_body": {
        "args": {"path": "sentence-transformers/stackexchange-duplicates", "data_dir": "body-body-pair"},
        "map_fn": lambda ex: {"anchor": ex["body1"], "positive": ex["body2"]},
        "loss": MultipleNegativesRankingLoss
    },
    "StackExchange_Duplicates_title_title": {
        "args": {"path": "sentence-transformers/stackexchange-duplicates", "data_dir": "title-title-pair"},
        "map_fn": lambda ex: {"anchor": ex["title1"], "positive": ex["title2"]},
        "loss": MultipleNegativesRankingLoss
    },
    "WikiAnswer_Pairs": {
        "args": {"path": "sentence-transformers/wikianswers-duplicates"},
        "map_fn": lambda ex: {"anchor": ex["anchor"], "positive": ex["positive"]},
        "loss": MultipleNegativesRankingLoss
    },
    "Natural_Questions": {
        "args": {"path": "sentence-transformers/natural-questions"},
        "map_fn": lambda ex: {"anchor": ex["query"], "positive": ex["answer"]},
        "loss": MultipleNegativesRankingLoss
    },
    "PAQ": {
        "args": {"path": "embedding-data/PAQ_pairs"},
        "map_fn": lambda ex: {"anchor": ex["set"][0], "positive": ex["set"][1]},
        "loss": MultipleNegativesRankingLoss
    },
    "Gooaq": {
        "args": {"path": "sentence-transformers/gooaq"},
        "map_fn": lambda ex: {"anchor": ex["question"], "positive": ex["answer"]},
        "loss": MultipleNegativesRankingLoss
    },
    "yahoo_title_answers": {
        "args": {"path": "sentence-transformers/yahoo-answers", "data_dir": "title-answer-pair"},
        "map_fn": lambda ex: {"anchor": ex["title"], "positive": ex["answer"]},
        "loss": MultipleNegativesRankingLoss
    },
    "msmacro_triplet": {
        "args": {"path": "sentence-transformers/msmarco-msmarco-MiniLM-L6-v3", "data_dir": "triplet-hard"},
        "map_fn": lambda ex: {"anchor": ex["query"], "positive": ex["positive"], "negative": ex["negative"]},
        "loss": MultipleNegativesRankingLoss
    },
    "trivia_qa_triplet": {
        "args": {"path": "sentence-transformers/trivia-qa-triplet", "data_dir": "triplet-all"},
        "map_fn": lambda ex: {"anchor": ex["anchor"], "positive": ex["positive"], "negative": ex["negative"]},
        "loss": MultipleNegativesRankingLoss
    },
    "nli_for_simcse_triplet": {
        "args": {"path": "sentence-transformers/nli-for-simcse", "data_dir": "triplet-all"},
        "map_fn": lambda ex: {"anchor": ex["anchor"], "positive": ex["positive"], "negative": ex["negative"]},
        "loss": MultipleNegativesRankingLoss
    },
    "quora_dup_triplet": {
        "args": {"path": "sentence-transformers/quora-duplicates", "data_dir": "triplet-all"},
        "map_fn": lambda ex: {"anchor": ex["anchor"], "positive": ex["positive"], "negative": ex["negative"]},
        "loss": MultipleNegativesRankingLoss
    }
}


stackexhange_title_best_answers = get_all_data_subset(
        "StackExchange_title_best_answer", 
        "flax-sentence-embeddings/stackexchange_title_best_voted_answer_jsonl", 
        "title_body", "upvoted_answer", MultipleNegativesRankingLoss)
stackexhange_titlebody_best_answers = get_all_data_subset(
    "StackExchange_titlebody_best_answer",
    "flax-sentence-embeddings/stackexchange_titlebody_best_voted_answer_jsonl",
    "title_body", "upvoted_answer", MultipleNegativesRankingLoss)
dataset_names = {**dataset_names, **stackexhange_title_best_answers, **stackexhange_titlebody_best_answers}

# # dataset_names = {**dataset_names, **stackexhange_title_best_answers, **stackexhange_titlebody_best_answers}
# # 1. Load & split each dataset
# dataset_dict = {}
# for name, config in dataset_names.items():
#     raw = load_dataset(**config["args"], 
#                        trust_remote_code=True,
#                        split=f"train[:{NROWS}]"
#                     )


#     if isinstance(raw, DatasetDict):
#         # concatenate all splits (train, validation, test, etc.)
#         raw = concatenate_datasets(list(raw.values()))

#     mapped = raw.map(config["map_fn"], remove_columns=raw.column_names)

#     # force the correct column order:
#     cols = ["anchor", "positive"]
#     if "negative" in mapped.column_names:
#         cols.append("negative")

#     mapped = mapped.select_columns(cols)

#     dataset_dict[name] = split_dataset(mapped, val_frac=0.1, seed=42)





In [5]:
# 1. Load & split each dataset
dataset_dict = {}
for name, config in dataset_names.items():
    print("*"*10 + f"Processing: {name}"+ "*"*10)
    out_dir = os.path.join(CACHE_DIR, name)
    if os.path.isdir(out_dir):
        # 1) cached splits already exist → load them
        print(f"Loading cached splits for {name} from {out_dir}")
        splits: DatasetDict = load_from_disk(out_dir)
    else:
        raw = load_dataset(
            **config["args"], 
            trust_remote_code=True,
            # split=f"train[:{NROWS}]"
            )

        if isinstance(raw, DatasetDict):
            # concatenate all splits (train, validation, test, etc.)
            raw = concatenate_datasets(list(raw.values()))

        mapped = raw.map(config["map_fn"], remove_columns=raw.column_names, num_proc=os.cpu_count() // 2)

        # force the correct column order:
        cols = ["anchor", "positive"]
        if "negative" in mapped.column_names:
            cols.append("negative")

        mapped = mapped.select_columns(cols)

        splits = split_dataset(mapped, val_frac=val_frac, test_frac=val_frac, seed=42)
        os.makedirs(out_dir, exist_ok=True)
        splits.save_to_disk(out_dir)
    
    # finally, record it
    dataset_dict[name] = splits

**********Processing: squad_v2**********
Loading cached splits for squad_v2 from ../data/stage1_cache/NROWS_None/squad_v2
**********Processing: wikipedia**********
Loading cached splits for wikipedia from ../data/stage1_cache/NROWS_None/wikipedia
**********Processing: StackExchange_Math_titlebody_answer**********
Loading cached splits for StackExchange_Math_titlebody_answer from ../data/stage1_cache/NROWS_None/StackExchange_Math_titlebody_answer
**********Processing: StackExchange_Math_title_answer**********
Loading cached splits for StackExchange_Math_title_answer from ../data/stage1_cache/NROWS_None/StackExchange_Math_title_answer
**********Processing: StackExchange_title_body**********
Loading cached splits for StackExchange_title_body from ../data/stage1_cache/NROWS_None/StackExchange_title_body
**********Processing: StackExchange_Duplicates_titlebody_titlebody**********
Loading cached splits for StackExchange_Duplicates_titlebody_titlebody from ../data/stage1_cache/NROWS_None/Stac

In [8]:
# 2. Build train/validation maps
train_dataset = {
    name: ds["train"]
    for name, ds in dataset_dict.items()
}

eval_dataset = {
    name: ds["validation"]
    for name, ds in dataset_dict.items()
}

test_dataset = {
    name: ds["test"]
    for name, ds in dataset_dict.items()
}

In [9]:
eval_dataset

{'squad_v2': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 7110
 }),
 'wikipedia': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 320391
 }),
 'StackExchange_Math_titlebody_answer': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 55048
 }),
 'StackExchange_Math_title_answer': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 55048
 }),
 'StackExchange_title_body': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 1266666
 }),
 'StackExchange_Duplicates_titlebody_titlebody': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 12526
 }),
 'StackExchange_Duplicates_body_body': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 12523
 }),
 'StackExchange_Duplicates_title_title': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 15226
 }),
 'WikiAnswer_Pairs': Dataset({
     features: ['anchor', 'positive'],
     num_rows: 38068979
 }),
 'Natural_Questions': Dataset({
     f

# Traning

In [10]:
# # this didnt work
# model_name = "/rhome/sawale/indus_traning/sentense_transformers/model_exploration/local_test_models/models--nasa-impact--nasa-smd-ibm-v0.1/snapshots/79f80ccf011700ec8d7cfbff2c5cae317d5f061a"
# model = SentenceTransformer(model_name)
model_name = "nasa-impact/nasa-smd-ibm-v0.1"
model = SentenceTransformer("nasa-impact/nasa-smd-ibm-v0.1", tokenizer_kwargs={"model_max_length": 512})

No sentence-transformers model found with name nasa-impact/nasa-smd-ibm-v0.1. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at nasa-impact/nasa-smd-ibm-v0.1 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
# # Load a model to train/finetune

# # model_name = "nasa-impact/nasa-smd-ibm-v0.1"

# model_name = "nasa-impact/indus-sde-v0.2"


# from sentence_transformers import SentenceTransformer, models

# # 1) load Transformer module straight from your checkpoint
# word_embedding_model = models.Transformer(
#     model_name_or_path=model_name,
#     max_seq_length=1024
# )

# # 2) add pooling
# pooling_model = models.Pooling(
#     word_embedding_model.get_word_embedding_dimension(),
#     pooling_mode_mean_tokens=True
# )

# # 3) build SentenceTransformer
# model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
# model

In [11]:
total_rows = sum([v.num_rows for k, v in train_dataset.items()]) + sum([v.num_rows for k, v in eval_dataset.items()])
OUTPUT_DIR = f"tmp_models/{model_name.split('/')[-1]}/{total_rows}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir=f"{OUTPUT_DIR}/checkpoints/",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    warmup_ratio=0.1,
    fp16=True,  # Set to False if your GPU can't handle FP16
    bf16=False,  # Set to True if your GPU supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # Losses using "in-batch negatives" benefit from no duplicates
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
    # use_cpu=True,
    # run_name="mpnet-base-all-nli-triplet",  # Used in W&B if `wandb` is installed
)


Currently using DataParallel (DP) for multi-gpu training, while DistributedDataParallel (DDP) is recommended for faster training. See https://sbert.net/docs/sentence_transformer/training/distributed.html for more information.


In [21]:
for i in eval_dataset.values():
    print(len(i["anchor"]))
    break

7110


In [ ]:
MAX_PER_SPLIT = 10

queries, corpus, relevant_docs = {}, {}, {}
q_id, c_id = 0, 0

for ds_name, ds in eval_dataset.items():
    ds = ds.select(range(MAX_PER_SPLIT))
    for ex in ds:
        q_id_str = f"{ds_name}-{q_id}"
        c_id_str = f"{ds_name}-{c_id}"

        queries[q_id_str] = ex["anchor"]
        corpus[c_id_str] = ex["positive"]

        # assuming the relevant document is the corresponding one (1-to-1)
        relevant_docs[q_id_str] = {c_id_str}

        q_id += 1
        c_id += 1


ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="ir_evaluator",
    batch_size=32,
    mrr_at_k=[1,5,10],
    ndcg_at_k=[10],
    accuracy_at_k=[1,3,5],
    precision_recall_at_k=[1,5,10],
    map_at_k=[100],
    show_progress_bar=True,
    write_csv=True
)



# for TripletEvaluator 
# 1) Gather three lists across all eval_dataset splits that have negatives
anchors   = []
positives = []
negatives = []

for ds in eval_dataset.values():
    # only consider splits with a “negative” column
    if "negative" not in ds.column_names:
        continue

    anchors   += ds["anchor"][:MAX_PER_SPLIT]
    positives += ds["positive"][:MAX_PER_SPLIT]
    negatives += ds["negative"][:MAX_PER_SPLIT]

# 2) Build the TripletEvaluator via lists
triplet_evaluator = TripletEvaluator(
    anchors=anchors,
    positives=positives,
    negatives=negatives,
    name="triplet_evaluator",
    batch_size=32,
    show_progress_bar=True,
    write_csv=True
)

# 4) Combine them into a SequentialEvaluator
seq_evaluator = SequentialEvaluator(
    [triplet_evaluator, ir_evaluator],
)



In [25]:
anchors

['Which member of the Monkees came from Washington DC?',
 "Who had a 70s No 1 hit with Billy, Don't Be A Hero?",
 'Which James Bond film features a song by Louis Armstrong?',
 'Who wrote the novel Evening Class?',
 'Who along with Philips developed the CD in the late 70s?',
 'Man In The Mirror first featured on which Michel Jackson album?',
 'Which oil scandal hit the US in 1924?',
 'Who was the first woman to make a solo flight across the Atlantic?',
 'Which British general was killed at Khartoum in 1885?',
 'Which parallel was the truce line in the Korean War?']

# Trainer

In [15]:
from datasets import load_dataset
from sentence_transformers import (
    SentenceTransformerTrainer,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss={k: v.get("loss")(model) for k, v in dataset_names.items()},
    evaluator=seq_evaluator,
)

# model.to('cpu')
trainer.train()

Step,Training Loss,Validation Loss


TrainOutput(global_step=4, training_loss=2.8469927310943604, metrics={'train_runtime': 8.2613, 'train_samples_per_second': 21.788, 'train_steps_per_second': 0.484, 'total_flos': 0.0, 'train_loss': 2.8469927310943604, 'epoch': 1.0})

In [16]:
seq_evaluator(model)


Batches: 100%|██████████| 1/1 [00:00<00:00,  6.88it/s]


{'multi-dataset-triplets_cosine_accuracy': 0.4000000059604645,
 'multi-dataset-pairs_cosine_accuracy@1': 0.5,
 'multi-dataset-pairs_cosine_accuracy@3': 0.8,
 'multi-dataset-pairs_cosine_accuracy@5': 0.8,
 'multi-dataset-pairs_cosine_precision@1': 0.5,
 'multi-dataset-pairs_cosine_precision@5': 0.16000000000000006,
 'multi-dataset-pairs_cosine_precision@10': 0.10000000000000002,
 'multi-dataset-pairs_cosine_recall@1': 0.5,
 'multi-dataset-pairs_cosine_recall@5': 0.8,
 'multi-dataset-pairs_cosine_recall@10': 1.0,
 'multi-dataset-pairs_cosine_ndcg@10': 0.7408894618915401,
 'multi-dataset-pairs_cosine_mrr@1': 0.5,
 'multi-dataset-pairs_cosine_mrr@5': 0.6333333333333333,
 'multi-dataset-pairs_cosine_mrr@10': 0.6600595238095238,
 'multi-dataset-pairs_cosine_map@100': 0.6600595238095238,
 'sequential_score': 0.7408894618915401}

In [ ]:
# Save the trained model
model.save_pretrained(f"{OUTPUT_DIR}/final_name")

# (Optional) Push it to the Hugging Face Hub
# model.push_to_hub("mpnet-base-all-nli-triplet")
